In [ ]:
import os
import re

import pypdf
import anthropic
from langchain.chains import RetrievalQA

In [ ]:
# Basic setup/config items
DOC_DIR = "../documents/Credit"
FILE_NAME = "sec.gov_Archives_edgar_data_1701758_000121390018004741_fs12018ex10-1_thelovesac.htm.pdf"
# LLM_MODEL_NAME = "claude-3-5-sonnet-20241022"
LLM_MODEL_NAME = "claude-opus-4-20250514"

In [ ]:
# https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm
details_to_extract = [
    'Parties involved (lenders, agent, borrower)',
    'Parties details (address, business, description)',
    'Terms (interest charged, rates, fees, payments, liens)',
    'Loan details (credit limit, letter of credit, repayments, timeline)'
    'Payments (agent clawback, sharing of payments, settlement among lenders)',
    'Miscellaneous (litigation, insurance, taxes)'
]

In [ ]:
# Function to read the PDF
def get_llm_text(pdf_file):
    reader = pypdf.PdfReader(pdf_file)
    text = "\n".join([page.extract_text() for page in reader.pages])

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove page numbers
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    return text

In [ ]:
pdf = (os.path.join(DOC_DIR, FILE_NAME))
document_text = get_llm_text(pdf)
print(document_text[:50])

In [ ]:
# Initialize the Anthropic client
client = anthropic.Anthropic()

def summarize_document(text, details_to_extract, model=LLM_MODEL_NAME, max_tokens=1000):
    # Format the details to extract to be placed within the prompt's context
    details_to_extract_str = '\n'.join(details_to_extract)

    # Prompt the model to summarize the sublease agreement
    prompt = f"""Summarize the following credit agreement. Focus on these key aspects:

    {details_to_extract_str}

    Provide the summary in bullet points nested within the XML header for each section. For example:

    <parties involved>
    - Lender: [Name]
    // Add more details as needed
    </parties involved>

    If any information is not explicitly stated in the document, note it as "Not specified". Do not preamble.

    Sublease agreement text:
    {text}
    """

    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system="You are a legal analyst specializing in credit law, known for highly accurate and detailed summaries of credit agreements.",
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "Here is the summary of the credit agreement: <summary>"}
        ],
        stop_sequences=["</summary>"]
    )

    return response.content[0].text


sublease_summary = summarize_document(document_text, details_to_extract)
print(sublease_summary)

Feel Free to cross check the extracted data points against the source document available at SEC portal here:
https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm